# Method C: U-Net training on Colab
**Owner: Lingyu Wu.** Before running:
1. Runtime → Change runtime type → **GPU**.
2. Key icon (Secrets) → add `GITHUB_TOKEN` (GitHub classic token with `repo` scope) and turn on notebook access.
3. Put the dataset on Google Drive as `<DRIVE>/data/raw/images` and `<DRIVE>/data/raw/masks`.

Steps: clone the repository → load the fixed split → build tiles → train with early stopping → choose the threshold on validation data → save `unet.pt` and `params/C.json`.

Colab already has every package in `requirements.txt`, so nothing needs installing.

In [ ]:
import torch
print("PyTorch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime -> GPU)")

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO = "OWNER/crack-width-measurement"         # TODO: replace OWNER with the repository owner
DRIVE = "/content/drive/MyDrive/crack-task2"    # TODO: the team folder on Google Drive
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/repo"):
    !git clone -q https://{token}@github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo

In [ ]:
import sys
os.environ["CRACK_DATA_DIR"] = f"{DRIVE}/data/raw"   # set before importing src
sys.path.insert(0, "/content/repo")

from src import config, data, method_c, utils

utils.set_seed()
params = utils.load_params("C", method_c.DEFAULT_PARAMS)
split = data.load_split()
print({name: len(ids) for name, ids in split.items()}, params)

In [ ]:
import numpy as np
from torch.utils.data import DataLoader, Dataset


class TileDataset(Dataset):
    """Tiles of the given photographs with their masks and padding masks.

    TODO(Lingyu):
      * keep crack tiles plus a sample of background tiles (all tiles of every
        training photograph do not fit in Colab memory; or cache tiles on Drive)
      * training only: paired flips and 90-degree rotations, image-only
        brightness/contrast changes (no resizing: the pixel scale must stay)
      * normalise RGB exactly as method_c.segment_c does
    """

    def __init__(self, ids, params, train):
        self.train = train
        self.items = []  # (image tile, mask tile, valid tile)
        for image_id in ids:
            image, mask = data.load_pair(image_id)
            tiles, _, valid = data.make_tiles(image, params["tile"], params["overlap"])
            mask_tiles, _, _ = data.make_tiles(mask, params["tile"], params["overlap"])
            # TODO(Lingyu): choose which tiles to keep and append them to self.items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        raise NotImplementedError("TODO(Lingyu): return image (3, H, W), mask (1, H, W), valid (1, H, W)")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
net = method_c.build_unet(params).to(device)   # TODO(Lingyu): implement build_unet in src/method_c.py
optimiser = torch.optim.Adam(net.parameters(), lr=params["learning_rate"])


def loss_fn(logits, target, valid):
    """TODO(Lingyu): weighted binary cross-entropy + soft Dice, ignoring padding (valid == 0).
    Class weights come from the training masks only."""
    raise NotImplementedError

In [ ]:
train_loader = DataLoader(TileDataset(split["train"], params, train=True),
                          batch_size=params["batch_size"], shuffle=True)
val_loader = DataLoader(TileDataset(split["val"], params, train=False),
                        batch_size=params["batch_size"])

os.makedirs(f"{DRIVE}/models", exist_ok=True)
best_dice, patience, epochs_without_gain = 0.0, 10, 0
for epoch in range(1, 201):
    # TODO(Lingyu): one training epoch over train_loader (net.train(), loss_fn, optimiser)
    # TODO(Lingyu): validation Dice over val_loader (net.eval(), torch.no_grad())
    val_dice = 0.0
    print(f"epoch {epoch}: validation Dice {val_dice:.4f}")
    if val_dice > best_dice:
        best_dice, epochs_without_gain = val_dice, 0
        torch.save(net.state_dict(), f"{DRIVE}/models/unet.pt")   # keep the best checkpoint
    else:
        epochs_without_gain += 1
        if epochs_without_gain >= patience:   # stop after 10 epochs without gain
            break

In [ ]:
# TODO(Lingyu): load the best checkpoint, predict the validation photographs with
# method_c.segment_c at several thresholds and keep the one with the best validation
# F1. Never use the test split here.
params["threshold"] = 0.5
utils.save_params("C", params)
print(params)

## After training
1. Commit `params/C.json` (download it from `/content/repo/params`, or push from Colab).
2. `unet.pt` stays on the team Drive; everyone copies it to `models/unet.pt`.
3. The next cell writes the Method C validation masks and copies them to Drive for evaluation.

In [ ]:
!mkdir -p models && cp "{DRIVE}/models/unet.pt" models/unet.pt
!python scripts/run_method.py --method C --split val
!mkdir -p "{DRIVE}/results" && cp -r results/C "{DRIVE}/results/"